# 3 — Storage

Companion to **section 4**. Storage couples the *hours*: what the battery
does at midnight depends on what the price will be at six. The dual that
appears, $\theta_t$, is the shadow price of stored energy — what one more
MWh in the tank is worth right now.

From here the models are built from standard PyPSA components rather than
written out constraint by constraint — but every dual means exactly what it
meant in section 2.

Runtime: about ten seconds.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
note = next(p for p in [here, *here.parents] if (p / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
sys.path.insert(0, str(note / "pipeline"))   # the run scripts' helpers
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

print(f"note root: {note}")

## The instance

DK1's actual 2024 fleet — wind and solar at the capacities the profiles were
normalised by, the thermal plants from the vendored network, and the import
backstop of section 3 priced at what the neighbours charged hour by hour.
That is what `pipeline/run_storage.py` solves; here we borrow its helpers.

The horizon is a **contiguous** fortnight from 1 January: a state of charge
cannot be sampled every k-th hour the way section 3's dispatch could.

In [ ]:
from model import dispatch, storage
from run_dispatch_t import dk1_fleet, import_price, PROFILE_OF, YEAR

HOURS = 336

tech = dispatch.read_tech(PROCESSED / "technology_costs_small.csv")
capacity, tech = dk1_fleet(tech)           # adds the imports row to `tech`

profiles = pd.read_csv(PROCESSED / f"profiles_dk1_{YEAR}.csv",
                       index_col="time", parse_dates=True)
sample = profiles.iloc[:HOURS]
load = sample["load_mw"].rename("load")
availability = pd.DataFrame({t: sample[c] for t, c in PROFILE_OF.items()})
hourly_cost = import_price(load.index)    # c_imports,t from the neighbours' spot prices

capacity.round(0).to_frame("MW")

## The arbitrage band

A 1 GW battery with four hours of storage. It charges when the price is low
and discharges when it is high, and the state of charge is the mirror image
of the price path.

In [ ]:
n = storage.build_network(tech, capacity, load, availability,
                          storage_power_mw=1000.0, storage_hours=4,
                          hourly_cost=hourly_cost)
storage.solve(n)
price = n.buses_t.marginal_price["elec"]
soc = n.storage_units_t.state_of_charge["battery"]

fig, axes = plt.subplots(2, 1, figsize=(8, 4.6), sharex=True)
axes[0].plot(price.index, price.values, color="#0072B2", lw=1.2)
axes[0].set_ylabel("price (EUR/MWh)")
axes[1].fill_between(soc.index, soc.values, color="#009E73", alpha=0.6)
axes[1].set_ylabel("state of charge (MWh)")
fig.autofmt_xdate()
plt.show()

Full just before the expensive hours, empty just after the cheap ones. Note
the round-trip: with a one-way efficiency of 0.95 the battery only moves
energy when the spread covers the loss, which is why it sits still through
flat stretches.

## What storage does to the price

Compare with and without the battery. Storage does not lower the *average*
price much; it narrows the **distribution** — and that is what pays it.

In [ ]:
n0 = storage.build_network(tech, capacity, load, availability,
                           storage_power_mw=0.0, hourly_cost=hourly_cost)
storage.solve(n0)
p0 = n0.buses_t.marginal_price["elec"]

revenue = storage.storage_revenue(n)      # sales minus purchases, EUR over the fortnight
print(f"battery revenue over {HOURS} hours: {revenue/1e3:,.0f} kEUR")

pd.DataFrame({
    "without battery": [p0.mean(), p0.std(), p0.max() - p0.min()],
    "with battery":    [price.mean(), price.std(), price.max() - price.min()],
}, index=["mean (EUR/MWh)", "std dev (EUR/MWh)", "range (EUR/MWh)"]).round(2)

Section 4.2 of the note shows that in the long run this revenue exactly
covers the battery's annualised cost — the zero-profit condition of section 7
applied to a storage unit. `pipeline/run_storage.py` computes the yardstick
from the cost data of section 7; the notebook `06-investment.ipynb` is where
that data enters.

## Ramping: the cost of inflexibility

Section 4.3 adds a ramp constraint to the slow thermal plants. Its dual is
the shadow price of flexibility, and `storage.flexibility_cost` sums it over
the horizon. Tighten the ramp rate with and without the battery, and watch
the battery absorb what the plants can no longer do.

In [ ]:
from run_storage import RAMP_TECHS

rows = []
for rho in [1.0, 0.3, 0.1]:                       # share of capacity per hour
    ramp = None if rho >= 1.0 else {g: rho for g in RAMP_TECHS}
    for mw in [0.0, 1000.0]:
        m = storage.build_network(tech, capacity, load, availability,
                                  storage_power_mw=mw, storage_hours=4,
                                  hourly_cost=hourly_cost, ramp_rate=ramp)
        storage.solve(m)
        rows.append({"ramp rate": rho, "battery MW": mw,
                     "system cost (MEUR)": m.objective / 1e6,
                     "flexibility cost (kEUR)": storage.flexibility_cost(m) / 1e3})
pd.DataFrame(rows).set_index(["ramp rate", "battery MW"]).round(2)

## Your turn

1. Give the battery 8 hours of duration instead of 4. What happens to wind's
   capture price? (Section 4.4 of the note explains why the answer is
   disappointing: `dispatch_t.capture_price` is the helper.)
2. Extend the horizon to a full year (`HOURS = 8784`; a minute or so) and
   compare the revenue with the annual cost the note quotes. Does the
   fortnight in January overstate or understate what the battery earns?
3. The note's figure 9.6 compares a battery that sees the whole year with one
   that sees one day at a time. `run_storage.py` has the myopic loop
   (`run_foresight`); read it and explain why the gap is small for a
   four-hour battery and would not be for a reservoir.

In [ ]:
# Try it here.